In [4]:
# ============================================
# DATATHON - LIMPIEZA DE PUNTOS DE RECARGA DGT
# Baseline de estaciones existentes en España
# ============================================

import re
import hashlib
from datetime import datetime, timezone
import xml.etree.ElementTree as ET

import requests
import pandas as pd

# -------------------------------------------------
# 1) Fuente de datos
# -------------------------------------------------
XML_URL = "https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml"

# Si ya lo tienes descargado en raw, usa esta ruta y comenta la descarga remota
LOCAL_XML_PATH = None
# Ejemplo:
# LOCAL_XML_PATH = "data/raw/charging_points/electrolineras.xml"


# -------------------------------------------------
# 2) Helpers
# -------------------------------------------------
INVALID_NUMERIC_STRINGS = {"", "+", "-", ".", "+.", "-."}

def localname(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

def norm_text(x):
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    s = re.sub(r"\s+", " ", s)
    return s

def parse_float(value):
    if value is None:
        return None
    s = str(value).replace(",", ".").strip()
    s = re.sub(r"[^0-9+\-\.eE]", "", s)
    if s in INVALID_NUMERIC_STRINGS:
        return None
    try:
        return float(s)
    except ValueError:
        return None

def power_to_kw(raw_power):
    """
    Convierte a kW valores como:
    - 150
    - 150 kW
    - 50000 W
    - 0.15 MW
    """
    if raw_power is None:
        return None

    s = str(raw_power).strip().lower().replace(",", ".")
    num = parse_float(s)
    if num is None:
        return None

    if "mw" in s:
        return num * 1000.0
    if "w" in s and "kw" not in s:
        return num / 1000.0
    return num

def is_valid_coord(lat, lon):
    if lat is None or lon is None:
        return False
    return -90 <= lat <= 90 and -180 <= lon <= 180

def is_valid_coord_spain(lat, lon):
    if not is_valid_coord(lat, lon):
        return False
    return 27 <= lat <= 45 and -19 <= lon <= 5

def parse_iso8601_utc(raw_ts):
    if raw_ts is None:
        return None
    s = norm_text(raw_ts)
    if s is None:
        return None
    try:
        dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc).isoformat()
    except Exception:
        return s

def stable_site_key(national_identifier, site_name, lat, lon):
    if national_identifier:
        return f"ni::{national_identifier.strip().lower()}"

    parts = [
        norm_text(site_name) or "",
        "" if lat is None else f"{lat:.6f}",
        "" if lon is None else f"{lon:.6f}",
    ]
    raw = "||".join(parts).lower()
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]
    return f"hash::{digest}"

def iter_children_by_tag(elem, tag_name):
    for child in list(elem):
        if localname(child.tag) == tag_name:
            yield child

def iter_desc_by_tag(elem, tag_name):
    for child in elem.iter():
        if localname(child.tag) == tag_name:
            yield child

def first_desc(elem, tag_name):
    for child in elem.iter():
        if localname(child.tag) == tag_name:
            return child
    return None

def first_desc_text(elem, tag_name):
    node = first_desc(elem, tag_name)
    if node is None:
        return None
    return norm_text(node.text)

def extract_multilang_value(elem):
    """
    Extrae texto de estructuras multilenguaje si existen.
    """
    if elem is None:
        return None

    direct = norm_text(elem.text)
    if direct:
        return direct

    values = []
    for ch in elem.iter():
        txt = norm_text(ch.text)
        if txt:
            values.append(txt)

    if values:
        return values[0]
    return None

def round_coord(lat, lon, decimals=6):
    if lat is None or lon is None:
        return None, None
    return round(lat, decimals), round(lon, decimals)


# -------------------------------------------------
# 3) Cargar XML
# -------------------------------------------------
def load_xml_root(xml_url=XML_URL, local_path=LOCAL_XML_PATH, timeout=60):
    if local_path:
        return ET.parse(local_path).getroot(), {"source": "local_file", "path": local_path}

    response = requests.get(xml_url, timeout=timeout)
    response.raise_for_status()
    return ET.fromstring(response.content), {"source": "remote_url", "url": xml_url}

root, source_info = load_xml_root()
print("Loaded XML from:", source_info)


# -------------------------------------------------
# 4) Parseo del XML
# -------------------------------------------------
def parse_sites_from_xml(root):
    sites = []

    # Busca publicaciones / tablas de infraestructuras energéticas
    publication_nodes = [node for node in root.iter() if localname(node.tag) == "energyInfrastructureSite"]

    for site_node in publication_nodes:
        site_name = None
        operator_name = None
        address = None
        postcode = None
        national_identifier = None
        type_of_site = None
        service_facility_type = None
        lat = None
        lon = None
        last_updated = None

        # Campos simples frecuentes
        for child in site_node.iter():
            tag = localname(child.tag)

            if tag in {"siteName", "name"} and site_name is None:
                site_name = extract_multilang_value(child)

            elif tag in {"operatorName"} and operator_name is None:
                operator_name = extract_multilang_value(child)

            elif tag in {"nationalIdentifier"} and national_identifier is None:
                national_identifier = norm_text(child.text)

            elif tag in {"siteAddress", "address"} and address is None:
                address = extract_multilang_value(child)

            elif tag in {"postcode", "postalCode"} and postcode is None:
                postcode = norm_text(child.text)

            elif tag in {"siteType", "typeOfSite"} and type_of_site is None:
                type_of_site = norm_text(child.text)

            elif tag in {"serviceFacilityType"} and service_facility_type is None:
                service_facility_type = norm_text(child.text)

            elif tag in {"latitude"} and lat is None:
                lat = parse_float(child.text)

            elif tag in {"longitude"} and lon is None:
                lon = parse_float(child.text)

            elif tag in {"publicationTime", "lastUpdated", "versionTime"} and last_updated is None:
                last_updated = parse_iso8601_utc(child.text)

        # Conectores y potencia
        connector_powers_kw = []
        number_of_connections = 0

        for child in site_node.iter():
            tag = localname(child.tag).lower()
            text = norm_text(child.text)

            # contar conectores de forma más flexible
            if "connector" in tag or "socket" in tag:
                number_of_connections += 1

            # capturar potencia con lógica más amplia
            if "power" in tag:
                p = power_to_kw(text)
                if p is not None:
                    connector_powers_kw.append(p)

                total_power_kw = round(sum(connector_powers_kw), 3) if connector_powers_kw else None

                # Si no detecta conectores explícitos pero sí potencia, asumimos al menos 1
                if number_of_connections == 0 and total_power_kw is not None:
                    number_of_connections = 1

        site_key = stable_site_key(national_identifier, site_name, lat, lon)

        sites.append({
            "site_key": site_key,
            "national_identifier": national_identifier,
            "site_name": site_name,
            "operator_name": operator_name,
            "address": address,
            "postcode": postcode,
            "type_of_site": type_of_site,
            "service_facility_type": service_facility_type,
            "lat": lat,
            "lon": lon,
            "number_of_connections": number_of_connections if number_of_connections > 0 else None,
            "total_power_kw": total_power_kw,
            "last_updated": last_updated,
        })

    return sites

sites = parse_sites_from_xml(root)
print("Raw parsed sites:", len(sites))


# -------------------------------------------------
# 5) Limpieza básica
# -------------------------------------------------
for s in sites:
    s["coord_valid"] = is_valid_coord(s["lat"], s["lon"])
    s["coord_valid_spain"] = is_valid_coord_spain(s["lat"], s["lon"])
    s["site_name_norm"] = (norm_text(s.get("site_name")) or "").lower()
    s["operator_name_norm"] = (norm_text(s.get("operator_name")) or "").lower()
    s["address_norm"] = (norm_text(s.get("address")) or "").lower()


# -------------------------------------------------
# 6) Deduplicación exacta
# -------------------------------------------------
exact_key_to_best = {}

for s in sites:
    lat_r, lon_r = round_coord(s["lat"], s["lon"], decimals=6)

    dedup_key = (
        s.get("national_identifier") or "",
        s.get("site_name_norm") or "",
        s.get("operator_name_norm") or "",
        lat_r,
        lon_r,
        s.get("address_norm") or "",
    )

    existing = exact_key_to_best.get(dedup_key)

    if existing is None:
        exact_key_to_best[dedup_key] = s
        continue

    # Priorizamos:
    # 1) coordenadas válidas en España
    # 2) dirección más completa
    # 3) más conexiones
    # 4) más potencia
    cur_score = (
        int(bool(s.get("coord_valid_spain"))),
        len(s.get("address") or ""),
        s.get("number_of_connections") or 0,
        s.get("total_power_kw") or 0,
    )
    old_score = (
        int(bool(existing.get("coord_valid_spain"))),
        len(existing.get("address") or ""),
        existing.get("number_of_connections") or 0,
        existing.get("total_power_kw") or 0,
    )

    if cur_score > old_score:
        exact_key_to_best[dedup_key] = s

sites_dedup = list(exact_key_to_best.values())
print("After exact dedup:", len(sites_dedup))


# -------------------------------------------------
# 7) Dataset limpio base
# -------------------------------------------------
sites_clean_all = [
    s for s in sites_dedup
    if s.get("coord_valid_spain")
]

print("Clean sites with valid Spain coords:", len(sites_clean_all))


# -------------------------------------------------
# 8) DataFrame final relevante para el datathon
# -------------------------------------------------
sites_model_df = pd.DataFrame(sites_clean_all)[[
    "site_key",
    "site_name",
    "operator_name",
    "lat",
    "lon",
    "number_of_connections",
    "total_power_kw",
    "postcode",
    "address",
    "type_of_site",
    "service_facility_type",
    "last_updated",
]].copy()

sites_model_df = sites_model_df.sort_values(
    by=["operator_name", "site_name", "lat", "lon"],
    na_position="last"
).reset_index(drop=True)

print("sites_model_df shape:", sites_model_df.shape)
print(sites_model_df.head())


# -------------------------------------------------
# 9) Export útil para vuestro pipeline
# -------------------------------------------------
# Baseline completo limpio
sites_model_df.to_csv("charging_points_clean_baseline.csv", index=False)

# Versión mínima de features
final_features_df = sites_model_df[[
    "site_key",
    "lat",
    "lon",
    "number_of_connections",
    "total_power_kw"
]].copy()

final_features_df.to_csv("charging_points_final_features.csv", index=False)

print("Saved:")
print("- charging_points_clean_baseline.csv")
print("- charging_points_final_features.csv")

Loaded XML from: {'source': 'remote_url', 'url': 'https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml'}
Raw parsed sites: 12075
After exact dedup: 12072
Clean sites with valid Spain coords: 12072
sites_model_df shape: (12072, 12)
                 site_key                                   site_name  \
0  hash::592f9cea72b4e0f6        (Acceso Restringido) General Química   
1  hash::905a43e5208100ec        (PTE REPARACIÓN) CUESTA EL BOLÓN PR5   
2  hash::7fa723e5fb1aeaf0  (en reparacion) CARRETERA ALDEHUELA 67 (4)   
3  hash::7a6a1c6b707f052a                                  0889000215   
4  hash::2026b8ff1f805c2a                                  0889000315   

  operator_name        lat       lon  number_of_connections  total_power_kw  \
0          None  42.718964 -2.985155                     16         29440.0   
1          None  41.512016 -5.748489                      8         44000.0   
2          None  41.506645 -5.728588              